In [1]:
# ==============================================================================
# CELL 1: GOLD STAR SCHEMA - CORPORATE FINANCIALS & PEER BENCHMARKING
# ==============================================================================
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

# 1. Dimension Table: Companies (ApexFin + Public Peers)
dim_company_data = [
    ("APEX", "Apex Financial Technologies", "Target Firm", "Tier-2 Payment Infrastructure", "US"),
    ("PYPL", "PayPal Holdings, Inc.", "Peer", "Digital Payments & Checkout", "US"),
    ("XYZ",  "Block, Inc. (Square)", "Peer", "Merchant Acquiring & P2P", "US"),
    ("FI",   "Fiserv, Inc.", "Peer", "Enterprise Fintech & Core Banking", "US")
]
sdf_dim_company = spark.createDataFrame(
    dim_company_data, ["ticker", "company_name", "entity_role", "primary_segment", "hq_country"]
)
sdf_dim_company.write.format("delta").mode("overwrite").saveAsTable("gold_dim_company")

# 2. Load Silver Macro & Bronze SEC Peers
pdf_macro = spark.read.table("silver_macro_features").toPandas()
pdf_sec = spark.read.table("bronze_sec_peers").toPandas()

# Normalize SEC revenue & operating income into clean quarterly peer records
rev_tags = ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"]
pdf_rev = pdf_sec[pdf_sec["metric_tag"].isin(rev_tags)].copy()
pdf_rev = pdf_rev.sort_values("filed_date").drop_duplicates(
    subset=["ticker", "fiscal_year", "fiscal_period"], keep="last"
)
pdf_rev = pdf_rev[pdf_rev["fiscal_period"].isin(["Q1", "Q2", "Q3", "Q4"])]

pdf_opinc = pdf_sec[pdf_sec["metric_tag"] == "OperatingIncomeLoss"].copy()
pdf_opinc = pdf_opinc.sort_values("filed_date").drop_duplicates(
    subset=["ticker", "fiscal_year", "fiscal_period"], keep="last"
)

pdf_peers_merged = pd.merge(
    pdf_rev[["ticker", "fiscal_year", "fiscal_period", "val_usd"]],
    pdf_opinc[["ticker", "fiscal_year", "fiscal_period", "val_usd"]],
    on=["ticker", "fiscal_year", "fiscal_period"],
    how="left",
    suffixes=("_rev", "_opinc")
)

peer_rows = []
for _, r in pdf_peers_merged.iterrows():
    rev_m = round(r["val_usd_rev"] / 1e6, 2)
    op_m = round(r["val_usd_opinc"] / 1e6, 2) if pd.notnull(r["val_usd_opinc"]) else round(rev_m * 0.16, 2)
    margin_pct = round((op_m / rev_m) * 100, 2) if rev_m > 0 else 15.0
    peer_rows.append({
        "ticker": r["ticker"],
        "fiscal_year": int(r["fiscal_year"]),
        "fiscal_quarter": r["fiscal_period"],
        "segment": "Consolidated",
        "gpv_millions": round(rev_m * 42.0, 2),
        "net_revenue_millions": rev_m,
        "operating_income_millions": op_m,
        "operating_margin_pct": margin_pct,
        "take_rate_bps": round((rev_m / (rev_m * 42.0)) * 10000, 1)
    })

# 3. Generate ApexFin Segment Financials Driven by Real Macro Covariates
np.random.seed(42)
apex_rows = []
segments = [
    ("B2B Commercial Rails", 4800.0, 215.0, 24.5, -14.0),     # Sensitive to Fed Funds hikes
    ("Merchant Acquiring",   3400.0, 245.0, 19.8, -8.5),      # Sensitive to CPI inflation
    ("Consumer Remittance",  1800.0, 285.0, 17.2, -5.0)       # Higher take-rate, resilient volume
]

for idx, m in pdf_macro.iterrows():
    growth_trend = 1.0 + (idx * 0.018)
    rate_drag = 1.0 - (m["FEDFUNDS"] * 0.015)
    cpi_drag = 1.0 - (max(0, m["cpi_yoy_pct"] - 2.0) * 0.012)
    
    for seg_name, base_gpv, base_tr_bps, base_margin, rate_sens in segments:
        noise = np.random.normal(1.0, 0.018)
        gpv = round(base_gpv * growth_trend * rate_drag * cpi_drag * noise, 2)
        # Take-rate compresses slightly over recent quarters (-310 bps story)
        compression = (idx / len(pdf_macro)) * 28.0
        tr_bps = round(base_tr_bps - compression + np.random.normal(0, 2.0), 1)
        rev = round(gpv * (tr_bps / 10000.0), 2)
        op_margin = round(base_margin - (compression * 0.11) + (rate_sens * (m["fed_funds_diff1"] * 0.15)), 2)
        op_inc = round(rev * (op_margin / 100.0), 2)
        
        apex_rows.append({
            "ticker": "APEX",
            "fiscal_year": int(m["fiscal_year"]),
            "fiscal_quarter": m["fiscal_quarter"],
            "segment": seg_name,
            "gpv_millions": gpv,
            "net_revenue_millions": rev,
            "operating_income_millions": op_inc,
            "operating_margin_pct": op_margin,
            "take_rate_bps": tr_bps
        })

pdf_all_fin = pd.concat([pd.DataFrame(apex_rows), pd.DataFrame(peer_rows)], ignore_index=True)
pdf_fact_fin = pd.merge(
    pdf_all_fin,
    pdf_macro[["fiscal_year", "fiscal_quarter", "observation_date", "FEDFUNDS", "CPIAUCSL", "cpi_yoy_pct", "DGS10", "DGS2", "yield_spread_10y2y", "yield_curve_inverted", "UNRATE", "ns_level", "ns_slope", "fed_funds_diff1"]],
    on=["fiscal_year", "fiscal_quarter"],
    how="inner"
)

sdf_fact_fin = spark.createDataFrame(pdf_fact_fin) \
    .withColumn("observation_date", F.to_date("observation_date"))

sdf_fact_fin.write.format("delta").mode("overwrite").saveAsTable("gold_fact_corporate_financials")
print(f"Created gold_dim_company & gold_fact_corporate_financials ({sdf_fact_fin.count():,} rows).")

StatementMeta(, a75c4c7d-f80e-4d47-bf30-705b8c9452ff, 3, Finished, Available, Finished, False)

Created gold_dim_company & gold_fact_corporate_financials (159 rows).


In [2]:
# ==============================================================================
# CELL 2: GOLD STAR SCHEMA - FRAUD SURVEILLANCE & STRATIFIED ML TRAINING SET
# ==============================================================================
from pyspark.sql import functions as F

# 1. Dimension Table: Payment Channels
dim_channel_data = [
    ("TRANSFER", "Inter-Account Wire / P2P Transfer", "High", "Instant Rail"),
    ("CASH_OUT", "External Off-Ramp Liquidation", "Critical", "Settlement Rail"),
    ("PAYMENT",  "Merchant Acquiring POS / Gateway", "Low", "Card / Acquiring Rail"),
    ("CASH_IN",  "Inbound Liquidity Deposit", "Low", "Funding Rail"),
    ("DEBIT",    "Direct ACH / Card Debit", "Medium", "Clearing Rail")
]
sdf_dim_channel = spark.createDataFrame(
    dim_channel_data, ["type", "channel_description", "inherent_risk_tier", "rail_category"]
)
sdf_dim_channel.write.format("delta").mode("overwrite").saveAsTable("gold_dim_channel")

# 2. Load Enriched Silver Transactions (6.36M rows)
df_silver_tx = spark.read.table("silver_paysim_enriched")

df_fraud = df_silver_tx.filter(F.col("isFraud") == 1)
df_legit = df_silver_tx.filter(F.col("isFraud") == 0)

# 3. Create 100K Direct Lake Power BI Fact Table (All 8,213 Fraud + 91,787 Legit Sample)
df_legit_bi = df_legit.sample(withReplacement=False, fraction=0.0145, seed=42).limit(91787)
df_gold_bi = df_fraud.unionByName(df_legit_bi) \
    .withColumn("legacy_alert_outcome",
        F.when((F.col("isFraud") == 1) & (F.col("isFlaggedFraud") == 1), "True Positive (Caught)")
         .when((F.col("isFraud") == 1) & (F.col("isFlaggedFraud") == 0), "False Negative (Missed Fraud)")
         .when((F.col("isFraud") == 0) & (F.col("isFlaggedFraud") == 1), "False Positive (False Alarm)")
         .otherwise("True Negative (Clean)")
    )

df_gold_bi.write.format("delta").mode("overwrite").saveAsTable("gold_fact_fraud_surveillance")

# 4. Create 250K Stratified ML Training Dataset (All 8,213 Fraud + 241,787 Legit)
df_legit_ml = df_legit.sample(withReplacement=False, fraction=0.0385, seed=99).limit(241787)
df_ml_train = df_fraud.unionByName(df_legit_ml)

df_ml_train.write.format("delta").mode("overwrite").saveAsTable("gold_ml_fraud_train")

print(f"Created gold_fact_fraud_surveillance: {df_gold_bi.count():,} rows")
print(f"Created gold_ml_fraud_train: {df_ml_train.count():,} rows")

StatementMeta(, a75c4c7d-f80e-4d47-bf30-705b8c9452ff, 4, Finished, Available, Finished, False)

Created gold_fact_fraud_surveillance: 100,000 rows
Created gold_ml_fraud_train: 250,000 rows
